# Importing libraries and loading RFM data

In [1]:
import sqlite3
import pandas as pd

rfm = pd.read_csv('rfm_segments.csv')
rfm.shape

(5878, 10)

# Create an in-memory SQLite database and load the data into it

In [2]:
conn = sqlite3.connect(':memory:')
rfm.to_sql('rfm', conn, index=False, if_exists='replace')
print("Table loaded")

Table loaded


# recreating the segment summary with SQL

In [3]:
query = """
    SELECT Segment,
           COUNT(*) AS num_customers,
           ROUND(SUM(Monetary), 2) AS total_revenue,
           ROUND(AVG(Monetary), 2) AS avg_customer_value
    FROM rfm
    GROUP BY Segment
    ORDER BY total_revenue DESC
"""

result = pd.read_sql_query(query, conn)
result

,Segment,num_customers,total_revenue,avg_customer_value
0,Champions,1482,12024330.14,8113.58
1,Loyal Customers,1221,2510046.35,2055.73
2,At Risk,353,1090694.27,3089.79
3,Needs Attention,471,498690.04,1058.79
4,New Customers,443,392267.02,885.48
5,Lost,868,266764.97,307.33
6,Hibernating,592,208174.47,351.65
7,Promising,385,204349.74,530.78
8,Cannot Lose Them,63,179487.26,2849.00


# Adding percentage-of-total columns

In [4]:
query = """
    SELECT Segment,
           COUNT(*) AS num_customers,
           ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM rfm), 1) AS pct_of_customers,
           ROUND(SUM(Monetary), 2) AS total_revenue,
           ROUND(SUM(Monetary) * 100.0 / (SELECT SUM(Monetary) FROM rfm), 1) AS pct_of_revenue
    FROM rfm
    GROUP BY Segment
    ORDER BY total_revenue DESC
"""

result = pd.read_sql_query(query, conn)
result

,Segment,num_customers,pct_of_customers,total_revenue,pct_of_revenue
0,Champions,1482,25.2,12024330.14,69.2
1,Loyal Customers,1221,20.8,2510046.35,14.4
2,At Risk,353,6.0,1090694.27,6.3
3,Needs Attention,471,8.0,498690.04,2.9
4,New Customers,443,7.5,392267.02,2.3
5,Lost,868,14.8,266764.97,1.5
6,Hibernating,592,10.1,208174.47,1.2
7,Promising,385,6.5,204349.74,1.2
8,Cannot Lose Them,63,1.1,179487.26,1.0


Champions are at roughly 25.2% of customers but contribute to 69.2% of revenue,

# Finding the win-back target list

In [5]:
query = """
    SELECT "Customer ID", Segment, Recency, Frequency, ROUND(Monetary,2) AS Monetary
    FROM rfm
    WHERE Segment IN ('At Risk', 'Cannot Lose Them')
    ORDER BY Monetary DESC
    LIMIT 15
"""

winback = pd.read_sql_query(query, conn)
winback

,Customer ID,Segment,Recency,Frequency,Monetary
0,12346,At Risk,326,12,77556.46
1,16754,At Risk,372,29,65500.07
2,13093,At Risk,276,55,54144.25
3,17850,At Risk,372,155,51208.87
4,13902,At Risk,632,5,34095.26
5,12482,At Risk,576,29,23691.40
6,14063,At Risk,438,9,22710.20
7,15808,At Risk,306,21,17180.25
8,17448,At Risk,497,46,14523.67
9,15760,Cannot Lose Them,631,2,13916.34


customer 12346 tops the list with £77,556.46 in historical spend and 326 days of silence this is a customer that needs to be won back

# Quantifying the concentration using The Pareto check

In [6]:
query = """
    SELECT
        SUM(CASE WHEN Segment IN ('Champions','Loyal Customers') THEN Monetary ELSE 0 END) AS top_segment_revenue,
        SUM(Monetary) AS total_revenue,
        ROUND(SUM(CASE WHEN Segment IN ('Champions','Loyal Customers') THEN Monetary ELSE 0 END) * 100.0 / SUM(Monetary), 1) AS pct_of_total
    FROM rfm
"""

pareto = pd.read_sql_query(query, conn)
pareto

,top_segment_revenue,total_revenue,pct_of_total
0,1.453438e+07,1.737480e+07,83.7


83.7% of total revenue comes from just Champions and Loyal Customers.

# R-Score distribution

In [7]:
query = """
    SELECT R_Score, COUNT(*) AS num_customers, ROUND(AVG(Monetary),2) AS avg_monetary
    FROM rfm
    GROUP BY R_Score
    ORDER BY R_Score DESC
"""

r_dist = pd.read_sql_query(query, conn)
r_dist

,R_Score,num_customers,avg_monetary
0,5,1188,7894.25
1,4,1176,2991.11
2,3,1167,1915.23
3,2,1172,1190.94
4,1,1175,721.73


customers with R_Score 5 (most recent) have a much higher average monetary value than those with R_Score 1 (least recent)

# Closing the connection

In [8]:
conn.close()
print("Connection closed")

Connection closed
